# 🧮 Étape 2 — Filtrage Linguistique & Formule Mathématique de Scoring ($S$) 💻 *(Exécution Locale)*

**Environnement** : Ce notebook tourne **100% en local sur ton PC (CPU)**.

**Objectif pédagogique** : Implémenter en Python la formule de détection du Camfranglais :
$$S = \mathbb{I}(N \ge 6) \times D_{\text{cam}} \times P_{\text{matrice}}$$
à partir de ton fichier [`../config/lexicon.json`](../config/lexicon.json) et l'utiliser pour filtrer tes données locales (`../conversations_dataset.json` et `../data/raw/raw_comments.jsonl`).

> 📖 **Conseil** : Consulte `02_filtering_scoring/README.md` pour le détail mathématique de chaque terme.

In [1]:
import re
import json
from pathlib import Path
from difflib import SequenceMatcher
from typing import List, Dict, Tuple

PROJECT_ROOT = Path("..").resolve()
LEXICON_PATH = PROJECT_ROOT / "config" / "lexicon.json"

with open(LEXICON_PATH, "r", encoding="utf-8") as f:
    LEXICON = json.load(f)

print(f"✅ Lexique chargé depuis {LEXICON_PATH}")
print(f"   - Marqueurs forts (w=2.0) : {len(LEXICON['strong_markers'])} mots")
print(f"   - Marqueurs moyens (w=1.0) : {len(LEXICON['medium_markers'])} mots")

✅ Lexique chargé depuis C:\Users\willi\Documents\Projet_Camfranglais\CamfranglaisLLM\config\lexicon.json
   - Marqueurs forts (w=2.0) : 54 mots
   - Marqueurs moyens (w=1.0) : 31 mots


---
## 1. Étage 1 : Nettoyage Structurel & Tokenisation Alphabétique (`TODO 1`)

Avant de calculer le score d'un commentaire :
1. On retire les URLs, les mentions `@user`, et les hashtags `#tag`.
2. On réduit les répétitions de lettres ou d'émojis (`🤣🤣🤣` $\rightarrow$ `🤣`).
3. On extrait la liste des **tokens alphabétiques** en minuscules pour compter $N$ (et vérifier $N \ge 6$).

In [9]:
# === EXERCICE GUIDÉ : TODO 1 ===
def normalize_and_tokenize(text: str) -> Tuple[str, List[str]]:
    """
    Nettoie le texte brut (en conservant un exemplaire des émojis pour le style)
    et retourne :
      - `cleaned_text` : le texte propre prêt pour l'entraînement
      - `words` : la liste des mots alphabétiques en minuscules (pour calculer N, D_cam et P_matrice)
    """
    if not text:
        return "", []

    cleaned = str(text)
    cleaned = re.sub(r'http\S+|www\.\S+', '', cleaned)
    cleaned = re.sub(r'@\s*[^\s@]+', '', cleaned)
    cleaned = re.sub(r'#\S+', '', cleaned)
    cleaned = re.sub(r'(.)\1{2,}', r'\1\1', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    # TODO 1 : Extraire tous les mots alphabétiques en minuscules de `cleaned`.
    # Astuce : utilise `re.findall(r"[a-zA-ZÀ-ÿ']+", cleaned.lower())`
    words = []
    words = re.findall(r"[a-zA-ZÀ-ÿ]+", cleaned.lower())
    
    # --------------------

    return cleaned, words

In [10]:
# 🧪 Cellule de validation du TODO 1
clean_txt, tokens = normalize_and_tokenize("Mdr @jean regarde https://t.co/xyz le mbom a ndem 🤣🤣🤣🤣 #237 #cameroun")
assert "@jean" not in clean_txt and "#237" not in clean_txt
assert tokens == ["mdr", "regarde", "le", "mbom", "a", "ndem"], f"Tokens inattendus : {tokens}"
print("🎉 TODO 1 validé !")

🎉 TODO 1 validé !


---
## 2. Étage 2 : Densité Lexicale Pondérée ($D_{\text{cam}}$) avec Tolérance Phonétique (`TODO 2`)

Sur TikTok, YouTube ou WhatsApp, un même mot de Camfranglais peut s'écrire `wanda`, `awanda`, `wanday` ou `tchop`, `tchopper`.
Complète la fonction `compute_d_cam(words, lexicon)` :
$$D_{\text{cam}} = \frac{\sum_{i=1}^{N} w_i}{N}$$

In [ ]:
# === EXERCICE GUIDÉ : TODO 2 ===
def get_word_camfranglais_weight(word: str, lexicon: Dict) -> float:
    """
    Retourne le poids Camfranglais d'un mot :
      - 2.0 si c'est un marqueur fort (ou une variante phonétique d'un marqueur fort)
      - 1.0 si c'est un marqueur moyen (ou une variante phonétique d'un marqueur moyen)
      - 0.0 sinon
    """
    w_strong = lexicon.get("strong_markers_weight", 2.0)
    w_medium = lexicon.get("medium_markers_weight", 1.0)
    strong_set = set(m.strip().lower() for m in lexicon["strong_markers"])
    medium_set = set(m.strip().lower() for m in lexicon["medium_markers"])
    french_set = set(m.strip().lower() for m in lexicon["french_anchors"])

    if word in french_set:
        return 0.0

    if word in strong_set:
        return w_strong
    if word in medium_set:
        return w_medium

    if len(word) >= 4:
        for root in strong_set:
            if len(root) >= 4 and (root in word or SequenceMatcher(None, word, root).ratio() >= 0.84):
                return w_strong
        for root in medium_set:
            if len(root) >= 4 and (root in word or SequenceMatcher(None, word, root).ratio() >= 0.84):
                return w_medium

    return 0.0


def compute_d_cam(words: List[str], lexicon: Dict) -> float:
    """
    Calcule la densité lexicale pondérée : D_cam = (somme des w_i) / N
    """
    if not words:
        return 0.0

    # TODO 2 : Calcule la somme des poids de chaque `w` dans `words` via `get_word_camfranglais_weight`
    # et divise par `len(words)`.
    
    d_cam = sum(get_word_camfranglais_weight(word, lexicon) for word in words) / len(words)
    return d_cam
    

In [19]:
# 🧪 Cellule de validation du TODO 2
assert get_word_camfranglais_weight("mbom", LEXICON) == 2.0
assert get_word_camfranglais_weight("awanda", LEXICON) >= 1.0
d_test = compute_d_cam(["le", "mbom", "a", "tchop"], LEXICON)
assert abs(d_test - 0.75) < 1e-5, f"Attendu 0.75, obtenu {d_test}"
print(f"🎉 TODO 2 validé ! Densité calculée = {d_test:.2f}")

🎉 TODO 2 validé ! Densité calculée = 0.75


---
## 3. Étage 3 : Le *Syntax Anchor* ($P_{\text{matrice}}$) et la Formule Globale ($S$) (`TODO 3`)

Pour rejeter le Pidgin pur (*"I di wanda how that masa go tchop all the money"*) et garder le Camfranglais (*"Je wanda comment le gars va tchop tout l'argent"*), on calcule :
$$P_{\text{matrice}} = \frac{C_{\text{fr}}}{C_{\text{fr}} + C_{\text{en/pid}} + \epsilon}$$
Puis le score final :
$$S = \mathbb{I}(N \ge 6) \times D_{\text{cam}} \times P_{\text{matrice}}$$

In [30]:
# === EXERCICE GUIDÉ : TODO 3 ===
def compute_p_matrice(words: List[str], lexicon: Dict, epsilon: float = 1e-5) -> float:
    """
    Calcule le ratio d'ancrage syntaxique français face aux ancres Pidgin/Anglais :
    P_matrice = C_fr / (C_fr + C_en_pid + epsilon)
    """
    fr_anchors = set(a.strip().lower() for a in lexicon["french_anchors"])
    en_pid_anchors = set(a.strip().lower() for a in lexicon["pidgin_english_anchors"])

    # TODO 3.1 : Compter combien de mots de `words` appartiennent à `fr_anchors` (c_fr)
    # et combien appartiennent à `en_pid_anchors` (c_en_pid), puis retourner la formule.
    c_fr = sum(word.strip().lower() in fr_anchors for word in words)
    c_en_pid = sum(word.strip().lower() in en_pid_anchors for word in words)
    
    return c_fr / (c_fr + c_en_pid + epsilon)
   


def score_camfranglais(text: str, lexicon: Dict, min_words: int = 6) -> Dict:
    """
    Calcule toutes les composantes du score S pour un commentaire donné.
    Retourne un dictionnaire détaillé : {'cleaned_text', 'N', 'd_cam', 'p_matrice', 'score', 'zone'}
    avec :
      - 'high' si score >= 0.15
      - 'grey' si 0.05 <= score < 0.15
      - 'low'  si score < 0.05
    """
    cleaned_text, words = normalize_and_tokenize(text)
    n_words = len(words)

    # TODO 3.2 : Calculer `d_cam`, `p_matrice` et `score = (1.0 if n_words >= min_words else 0.0) * d_cam * p_matrice`
    # Déterminer la `zone` ('high', 'grey', ou 'low').
    
    # --- TON CODE ICI ---
    d_cam = compute_d_cam(words, lexicon)
    p_matrice = compute_p_matrice(words, lexicon)
    
    if n_words >= min_words :
        score = 1.0 * d_cam * p_matrice
    else :
        score = 0.0
        
    if score < 0.05 :
        zone = "low"
    if 0.05 <= score < 0.15 :
        zone = "grey"
    if score >= 0.15 :
        zone = "high"
   

    return {
        "cleaned_text": cleaned_text,
        "N": n_words,
        "d_cam": round(d_cam, 4),
        "p_matrice": round(p_matrice, 4),
        "score": round(score, 4),
        "zone": zone
    }

In [31]:
# 🧪 Cellule de validation du TODO 3 : Le crash-test Pidgin vs Français vs Camfranglais !
test_sentences = [
    ("Pidgin pur (Doit être rejeté)", "I di wanda how that masa go tchop all the money"),
    ("Français standard (Doit être rejeté)", "Je me demande vraiment comment cet homme va dépenser tout cet argent."),
    ("Trop court < 6 mots (Doit être rejeté)", "Mdr le gars awanda"),
    ("Vrai Camfranglais (Doit être en Zone Haute)", "Je wanda comment le mbom va tchop tout le kolo au ngata")
]

for label, phrase in test_sentences:
    res = score_camfranglais(phrase, LEXICON)
    print(f"[{label}]\n  Phrase : {phrase}\n  Résultat : {res}\n")

assert score_camfranglais(test_sentences[0][1], LEXICON)["score"] == 0.0
assert score_camfranglais(test_sentences[1][1], LEXICON)["score"] == 0.0
assert score_camfranglais(test_sentences[2][1], LEXICON)["score"] == 0.0
assert score_camfranglais(test_sentences[3][1], LEXICON)["zone"] == "high"
print("🎉 Bravo ! Ta formule mathématique S distingue parfaitement le Camfranglais du Pidgin et du Français.")

[Pidgin pur (Doit être rejeté)]
  Phrase : I di wanda how that masa go tchop all the money
  Résultat : {'cleaned_text': 'I di wanda how that masa go tchop all the money', 'N': 11, 'd_cam': 0.3636, 'p_matrice': 0.0, 'score': 0.0, 'zone': 'low'}

[Français standard (Doit être rejeté)]
  Phrase : Je me demande vraiment comment cet homme va dépenser tout cet argent.
  Résultat : {'cleaned_text': 'Je me demande vraiment comment cet homme va dépenser tout cet argent.', 'N': 12, 'd_cam': 0.0, 'p_matrice': 1.0, 'score': 0.0, 'zone': 'low'}

[Trop court < 6 mots (Doit être rejeté)]
  Phrase : Mdr le gars awanda
  Résultat : {'cleaned_text': 'Mdr le gars awanda', 'N': 4, 'd_cam': 0.75, 'p_matrice': 1.0, 'score': 0.0, 'zone': 'low'}

[Vrai Camfranglais (Doit être en Zone Haute)]
  Phrase : Je wanda comment le mbom va tchop tout le kolo au ngata
  Résultat : {'cleaned_text': 'Je wanda comment le mbom va tchop tout le kolo au ngata', 'N': 12, 'd_cam': 0.6667, 'p_matrice': 1.0, 'score': 0.6667, 'zo

---
## 4. La Règle d'Or SFT & Filtrage de ton `../conversations_dataset.json` local (`TODO 4`)

Complète le **`TODO 4`** pour filtrer ton fichier existant `../conversations_dataset.json` et sauvegarder les paires certifiées Camfranglais dans `../data/processed/filtered_whatsapp_sft.json` !

In [37]:
# === EXERCICE GUIDÉ : TODO 4 ===
def filter_sft_conversations(conversations_list: List[Dict], lexicon: Dict, threshold: float = 0.15) -> List[Dict]:
    """
    Filtre une liste d'objets de la forme :
    {"conversations": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
    selon la Règle d'Or SFT :
      - Le message 'user' doit contenir au moins 3 mots.
      - Le message 'assistant' doit obtenir un score Camfranglais >= threshold.
    """
    valid_conversations = []

    for item in conversations_list:
        conv = item.get("conversations", [])
        if len(conv) < 2:
            continue
        user_msg = conv[0].get("content", "")
        assistant_msg = conv[1].get("content", "")

        # TODO 4 : Appliquer `normalize_and_tokenize(user_msg)` et `score_camfranglais(assistant_msg, lexicon)`.
        # Si le user_msg a >= 3 mots ET que le score de assistant_msg >= threshold,
        # ajouter la paire nettoyée dans `valid_conversations`.
        user_cleaned,user_words = normalize_and_tokenize(user_msg)
        assistant_result = score_camfranglais(assistant_msg, lexicon)
        
        if len(user_words) >=3 and assistant_result["score"] >= threshold :
            valid_conversations.append({"conversations" : [{"role": "user", "content": user_cleaned},{"role": "assistant","content": assistant_result["cleaned_text"]}]})
        

        # --------------------

    return valid_conversations

In [38]:
# 🧪 Cellule de validation du TODO 4 sur ton fichier ../conversations_dataset.json local
conv_dataset_path = PROJECT_ROOT / "conversations_dataset.json"
if conv_dataset_path.exists():
    with open(conv_dataset_path, "r", encoding="utf-8") as f:
        existing_data = json.load(f)
    kept = filter_sft_conversations(existing_data, LEXICON, threshold=0.15)
    output_filtered_path = PROJECT_ROOT / "data" / "processed" / "filtered_whatsapp_sft.json"
    output_filtered_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_filtered_path, "w", encoding="utf-8") as f:
        json.dump(kept, f, indent=2, ensure_ascii=False)
    print(f"📊 Audit local de ../conversations_dataset.json : {len(kept)} paires Camfranglais retenues sur {len(existing_data)} !")
    print(f"💾 Sauvegardé dans : {output_filtered_path}")
print("🎉 Étape 2 locale validée ! Direction 03_dataset_builder.")

📊 Audit local de ../conversations_dataset.json : 179 paires Camfranglais retenues sur 3243 !
💾 Sauvegardé dans : C:\Users\willi\Documents\Projet_Camfranglais\CamfranglaisLLM\data\processed\filtered_whatsapp_sft.json
🎉 Étape 2 locale validée ! Direction 03_dataset_builder.
